## Issue 6b: Production ESMFold2 fold of the 90pid centroids

Lead     : `Purav Gupta`

Issue    : [Github Issue #6](https://github.com/petadex/igem-toronto/issues/6) — _fold the family-cluster centroids_

Start    : `2026-08-06`

Complete : `N/A`

Files    : `resources/260806_issue6b_foldrun/data/` — local working dir on Compute Canada `/project` (git-ignored)

S3 files : `s3://petadex-protein-structures/esmfold2-centroids/` — remote archive (synced after batches, not during)

---

### Data Accessed: 2026-06-21
Input is the 90pid catalytic-core FASTA already on disk (`90pid_catalytic_orfs.fa`, 18,172,960 records), derived earlier from SQL. That table was then saved at `s3://petabite/petadex/90pid_enzymes_clusters.csv`

```bash
cat 90pid_enzymes_clusters.csv | cut -f 2 -d ',' > 90_orfs.txt
aws s3 cp s3://petadex/logan/petadex.catalytic_orfs.v1.1.fa - --no-sign-request \
  | seqkit grep --id-regexp '^([^|]+)' -f 90_orfs.txt > 90pid_catalytic_orfs.fa
```

## Introduction

Issue 6 built the ESMFold2 folding pipeline; Issue 6a settled the hyperparameters (**68 diffusion steps / 10 refinement loops**). This notebook takes the full **90pid centroid** set — 18,172,960 catalytic-core sequences — and attempts to benchmarking folding on the Compute Canada / Vector **Killarney** cluster (L40S GPUs, `/project` with ~740 GB disk space free).

Rather than reuse the AWS-fleet driver from Issue 6, this run uses a fresh, self-contained script (`resources/260806_issue6b_foldrun/fold.py`) with no S3/spot machinery: load ESMFold2 once, fold a length-banded shard of the input, write structures locally. Output goes to `/project` first; S3 archival is a separate, later step.

## Objectives

1. Build the ORFid-keyed fold input from the existing catalytic-core FASTA.
2. Run a **100-sequence test on one L40S** to measure folding throughput and per-structure disk.
3. **Extrapolate** GPU-hours and total storage to all 18.17M centroids, and size the fleet.
4. Scale out via a SLURM job array (one length-band shard per GPU) and archive to S3.

---

## Materials and Methods

### Step 0: Environment

The petabite venv (Python 3.12, torch + Biohub `esm`/`transformers` forks) is activated by `activate_petabite.sh`, which also sets `PYTHONNOUSERSITE=1` (keeps `~/.local` from shadowing the venv) and loads the cluster modules. The HuggingFace cache lives on `/project` (home is quota-limited).

```bash
source activate_petabite.sh
export HF_HOME=/home/purav/projects/aip-wanglab/purav/hf_cache   # already in ~/.bashrc

# sanity: the ESMFold2 model class imports cleanly
python -c "from transformers.models.esmfold2.modeling_esmfold2 import ESMFold2Model; print('ok')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
```

> **Precision: fp32 required.** ESMFold2 wraps the 6.3B-param ESMC-6B backbone. bf16 *loads* fine (~13 GB peak VRAM) but its **fold forward pass errors on this build**: `RuntimeError: mat1 and mat2 must have the same dtype, but got Float and BFloat16` (an uncasted fp32 tensor meets a bf16 weight, every sequence, regardless of length). So we fold in **fp32** (~25 GB weights, fits the 46 GB L40S with room for activations on these short cores). `--dtype fp32` is the default in `fold.py`.


### Step 1: Build the fold input

`make_fold_input.sh` rewrites the catalytic-core FASTA headers so every record is keyed by its ORFid (`>3|7QJM|||||/95-371` → `>orf3`) — that `>orf<id>` becomes the output filename, so structures join back on ORFid. It then prints `seqkit stats`, from which we record **num_seqs** (expect 18,172,960) and **sum_len** (total residues — the key input to the time estimate).

```bash
cd resources/260806_issue6b_foldrun
./make_fold_input.sh
# -> data/90pid_catalytic_orfs.orfid.fa
# note the num_seqs and sum_len it prints; plug them into Step 3.
```

> rewriting headers: /home/purav/projects/aip-wanglab/purav/igem-toronto/90pid_catalytic_orfs.fa -> /home/purav/projects/aip-wanglab/purav/igem-toronto/resources/260806_issue6b_foldrun/data/90pid_catalytic_orfs.orfid.fa

> === seqkit stats (num_seqs should be 18,172,960; sum_len = total residues) ===

> file    format  type    num_seqs        sum_len min_len avg_len max_len

> /home/purav/projects/aip-wanglab/purav/igem-toronto/resources/260806_issue6b_foldrun/data/90pid_catalytic_orfs.orfid.fa

> FASTA   Protein 18172960       4998232825      29      275.0   59275

### Step 2: Test run — 100 sequences on 1 GPU

Grab one L40S, confirm the model loads (`--load-only`), then fold the first 100 records. `--s3` doesn't exist in this script — output is always local.

```bash
salloc --gres=gpu:1 --cpus-per-task=4 --mem=48G --time=1:00:00   # + your --account/--partition

cd resources/260806_issue6b_foldrun

# 1) confirm weights resolve + measure load/VRAM (no folding)
python fold.py data/90pid_catalytic_orfs.orfid.fa --outdir data/test100 --load-only
```

In [ ]:
# 2) fold 100 (bf16; fall back to --dtype fp32 only if bf16 errors)
python fold.py data/90pid_catalytic_orfs.orfid.fa --outdir data/test100 \
    --limit 100 --num-sampling-steps 68 --num-loops 10 --dtype fp32

This writes `data/test100/structures/*.cif`, `data/test100/metrics/*.json`, and `data/test100/progress.shard1of1.csv`. Re-running is a no-op (finished `.cif` files are skipped).

### Step 3: A representative, length-stratified sample

The 100-seq test folded only 29-34aa cores (`fold.py` sorts by length, so `--limit` takes the shortest). Fold cost is super-linear in length, so that throughput is ~20x too optimistic. Added `fold.py --sample`, which picks N sequences evenly spaced across the length range for an unbiased timing curve.

```bash
python fold.py data/90pid_catalytic_orfs.orfid.fa --outdir data/sample --sample 80 --max-len 2048
```


**Estimate.** `estimate_fleet.py` fits `wall = a * L^b` from the sample and integrates it over the true length histogram (throughput x total residues, not mean-time x count).

```bash
python estimate_fleet.py --test-dir data/sample --length-hist data/length_hist.tsv --max-len 2048
```


> **Note.** This first estimate used a provisional 2048aa cap and had not yet measured the real OOM ceiling or tested ESMFold2-Fast. The `calibrate.sbatch` job below did both in one unattended run, and the numbers were refined (see the calibration note).


### Step 4: Test ESMFold2-Fast

`biohub/ESMFold2-Fast` is not MSA-conditioned and has a lighter 24-layer trunk (vs 48). It relies directly on the protein language model embedding and reportedly matches ESMFold2 with little quality loss, so it should fold faster. Same model class, same fold API, same params, only the repo id changes (`fold.py --fast`).


These manual runs are what `calibrate.sbatch` bundles (stratified sample + ladder OOM probe + estimate + quality diff) into one unattended job, for both models.

```bash
# Fast on the same stratified 80 (seed 0 -> identical sequences as data/sample)
python fold.py data/90pid_catalytic_orfs.orfid.fa --outdir data/sample_fast --sample 80 --max-len 1280 --fast

# estimate + quality diff (mean pLDDT / pTM, full vs Fast on the shared seqs)
python estimate_fleet.py --test-dir data/sample_fast --length-hist data/length_hist.tsv --max-len 1280
```


### Calibration results (2026-08-07)

One `calibrate.sbatch` job on one L40S produced all of this: stratified samples for both models, a length-ladder OOM probe, the estimates, and a quality diff.

- **bf16 unusable** on this build (Float/BFloat16 matmul in the fold pass); **fp32 mandatory** (~25 GB).
- **fp32 OOM ceiling ~1300aa** (1285 folds, 1529 OOMs), same for both models (memory = shared ESMC-6B backbone + L^2 pair tensor). Cap `--max-len 1280` keeps 99.75%; OOMs cost ~25 min each, so cap below the ceiling.
- **Fast vs Full** (79 shared seqs): 1.78x faster (median 4.79s vs 8.52s) for -2.05 pLDDT (81.8 vs 83.9) and -0.022 pTM.

**Projection** (foldable <=1280aa: 18,128,069 seqs, 4.92 B residues):

| | Full | **Fast (chosen)** |
|---|---|---|
| GPU-hours | 66,001 | **36,034** |
| @64 / 128 GPUs | ~43 / 21.5 d | **~24 / 12 d** |
| storage | 3.15 TB | 3.15 TB (~5 offload waves) |

**Decision: ESMFold2-Fast** — nearly half the compute for ~2 pLDDT, advantage grows with length. The 44,891 seqs >1280aa (0.25%) are deferred to an H100 run.


### Step 5: Scale out (SLURM job array)

Locked config in `submit_fold.sh`: **`FAST=1`** (ESMFold2-Fast), **`NSHARDS=1000`** (Killarney `MaxArraySize`=1001), `--max-len 1280`, fp32. Each array task folds one **cost-balanced** shard (`select_shard` splits by `L^1.2` cost, not count, so every shard is ~37 GPU-h despite the 15× length skew) on one L40S. The 2-day partition (`gpubase_l40s_b4`) fits a shard in one job → single pass, no resubmits. `#SBATCH --account=aip-wanglab` is already set (matches `boltz_petase`).

```bash
cd resources/260806_issue6b_foldrun
TESTN=2 ./submit_fold.sh     # validate plumbing on 2 shards first
squeue --me
tail -f logs/fold90pid-<jobid>_1.out   # expect: loaded (biohub/ESMFold2-Fast) + folds ticking

./submit_fold.sh             # all 1000 shards; set MAXCONCURRENT to your allocation's GPU limit
```

**Resume is free**: a killed/preempted shard re-runs from where it stopped (finished `.cif` files are skipped). Wall-clock ≈ 36,069 GPU-h / MAXCONCURRENT (16→94d, 32→47d, 64→24d).

**Storage: ~5 waves.** 3.15 TB won't fit the 740 GB budget, so archive and clear per wave:
```bash
./sync_to_s3.sh          # dry-run
./sync_to_s3.sh --go     # upload structures/ + metrics/, then delete the local wave
```

**Long tail.** The 44,891 seqs >1280aa (0.25%) are skipped on L40S; fold them later on an H100 partition (80 GB fits the L² tensor).

OUT: `resources/260806_issue6b_foldrun/data/fold90pid/{structures,metrics}/`

### Step 6: Analysis (stub)

Downstream quality analysis (pLDDT/pTM distributions, comparison against experimental structures, FoldSeek novelty) to be added once a representative batch has folded.

---

## Results & Discussion

**Key metrics** (ESMFold2-Fast, fp32, L40S, 68 steps / 10 loops):
- Throughput fit `wall = 0.00812·L^1.202`; median fold 4.79 s; mean pLDDT 81.8, pTM 0.729.
- Disk: 641 B/residue (cif) + 194 B/seq (json).

**Projection** (foldable ≤1280aa: 18,128,069 seqs / 99.75%, 4.92 B residues):
- **36,034 GPU-hours** (~24 days on 64 L40S) and **3.15 TB** total.

**Observations**:
- **bf16 is broken on this build** — loads fine, but the fold pass hits a Float/BFloat16 matmul on every sequence. fp32 is mandatory (~25 GB, fits the 46 GB L40S).
- **fp32 OOM ceiling ≈ 1300aa** (1285 folds, 1529 OOMs), identical for Full and Fast — memory is dominated by the shared ESMC-6B backbone + the L² pair tensor. Cap at 1280 (keeps 99.75%); OOMs cost ~25 min each, so cap below the ceiling.
- **Length is the dominant cost driver**: 29–59,275aa, median 264, and cost ~L^1.2, so a naive first-N test (all 29–34aa) over-estimated throughput ~20×. Sharding is cost-balanced to avoid a 15× straggler.
- **Fast vs Full**: 1.78× faster for −2.05 pLDDT / −0.022 pTM. Chose Fast for the first-pass catalog.
- **Storage doesn't fit** the 740 GB budget → ~5 fold-then-offload waves to S3.

**Follow-up questions**:
- How many concurrent L40S does the `aip-wanglab` allocation allow? (sets wall-clock)
- Fold the 44,891 >1280aa sequences on H100 (80 GB), or leave deferred?
- Final S3 layout, and dedup against ESM Atlas prefolds before or after?